# 02 · Thử nghiệm và trực quan hóa tăng cường dữ liệu (Augmentation)

Mục tiêu của notebook:
1. **Hiểu rõ pipeline tăng cường dữ liệu**: Kiểm tra các phép biến đổi được định nghĩa trong `configs/augmentation.yaml` (HFlip, VFlip, ColorJitter, GaussianBlur, RandomScaleCrop, RandomCropFixed, ResizeLongest).
2. **Đồng bộ hóa 3 thành phần**: Đảm bảo **Ảnh - Bounding Box - Instance Mask** biến đổi nhất quán (không lệch box, không cắt mất quả mà vẫn giữ box).
3. **Sinh Density Map cho CSRNet**: Trực quan hóa density map ground-truth sinh ra sau biến đổi (tổng density map bảo toàn bằng số quả).
4. **Phân tích phân bố trước và sau Augmentation**: Đánh giá sự thay đổi về kích thước quả, số lượng quả mỗi ảnh sau Random Crop.
5. **So sánh với Augmentation của Ultralytics (YOLO)**: Mosaic, HSV, Flip.

> Có thể chạy trên môi trường máy cá nhân hoặc Kaggle (tự động fallback sang synthetic data nếu chưa có dataset thật).

In [ ]:
import os, sys, json, time, shutil, warnings
from pathlib import Path

# ---- Xác định thư mục gốc project: chạy được cả local lẫn Kaggle ----
IN_KAGGLE = Path('/kaggle/working').exists()
if IN_KAGGLE:
    CODE_INPUT = Path('/kaggle/input/fruitcount-code/FruitCount')   # sửa theo tên Kaggle Dataset chứa code
    ROOT = Path('/kaggle/working/FruitCount')
    if not ROOT.exists():
        shutil.copytree(CODE_INPUT, ROOT)
    OUT_BASE = Path('/kaggle/working')
else:
    ROOT = Path.cwd().resolve()
    while ROOT != ROOT.parent and not (ROOT / 'src').is_dir():
        ROOT = ROOT.parent
    OUT_BASE = ROOT / 'outputs'
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image as IPImage

from src.utils import load_yaml, save_yaml, read_json, write_json, load_image, ensure_dir

DS_CFG_PATH = Path(os.environ.get('FRUITCOUNT_DATASET_CFG', ROOT / 'configs/dataset.yaml'))
DS_CFG = load_yaml(DS_CFG_PATH)
PROCESSED = ROOT / DS_CFG.get('processed_dir', 'data/processed')
RUNS = Path(os.environ.get('FRUITCOUNT_RUNS', OUT_BASE / 'runs' if IN_KAGGLE else ROOT / 'runs'))
OUT_DIR = ensure_dir(OUT_BASE / 'notebooks' / '02_augmentation')   # hình + bảng của notebook này

warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False})
FAMILY_COLORS = {'detection': '#4C72B0', 'segmentation': '#55A868', 'density': '#C44E52'}


def savefig(fig, name):
    fig.savefig(OUT_DIR / f'{name}.png', dpi=150, bbox_inches='tight')
    plt.show()


class StopExecution(Exception):
    """Dừng 'Run All' một cách êm (không in traceback)."""
    def _render_traceback_(self):
        return []


try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x

print(f'Môi trường : {"Kaggle" if IN_KAGGLE else "Local"}')
print(f'ROOT       : {ROOT}')
print(f'PROCESSED  : {PROCESSED}')
print(f'RUNS       : {RUNS}')
print(f'OUT_DIR    : {OUT_DIR}')

## 0. Cấu hình & Tham số

In [ ]:
USE_SYNTHETIC_IF_MISSING = True
FORCE_RELOAD = False

AUG_CFG_PATH = ROOT / 'configs/augmentation.yaml'
AUG_CFG = load_yaml(AUG_CFG_PATH)

CSRNET_CFG_PATH = ROOT / 'configs/models/csrnet.yaml'
CSRNET_CFG = load_yaml(CSRNET_CFG_PATH)
DENSITY_CFG = CSRNET_CFG.get('density', {})

print("Augmentation config loaded from:", AUG_CFG_PATH)
print("Density config:", DENSITY_CFG)

## 1. Nạp dữ liệu mẫu

In [ ]:
import cv2

def find_dataset_dirs(base='/kaggle/input'):
    """Tìm cặp thư mục images/ + masks/ (ưu tiên đường dẫn có chữ 'train')."""
    base = Path(base)
    if not base.exists():
        return None, None
    cands = [d for d in base.rglob('images') if d.is_dir() and (d.parent / 'masks').is_dir()]
    cands.sort(key=lambda d: ('train' not in str(d).lower(), len(str(d))))
    return (cands[0], cands[0].parent / 'masks') if cands else (None, None)


def make_synthetic_dataset(root, n=60, size=(480, 640), seed=0):
    """Ảnh 'vườn' giả: quả tròn đỏ trên nền xanh, CÓ chồng lấn (quả sau đè quả trước) + instance mask."""
    rng = np.random.default_rng(seed)
    img_dir, mask_dir = ensure_dir(Path(root) / 'images'), ensure_dir(Path(root) / 'masks')
    h, w = size
    for k in range(n):
        img = np.full((h, w, 3), (40, 110, 40), np.float32) + rng.normal(0, 12, (h, w, 3))
        img = np.clip(img, 0, 255).astype(np.uint8)
        mask = np.zeros((h, w), np.uint16)
        for i in range(1, int(rng.integers(3, 70)) + 1):
            r = int(rng.integers(6, 18))
            cx, cy = int(rng.integers(r, w - r)), int(rng.integers(r, h - r))
            color = tuple(int(c) for c in (rng.integers(170, 235), rng.integers(20, 80), rng.integers(20, 60)))
            cv2.circle(img, (cx, cy), r, color, -1)
            cv2.circle(mask, (cx, cy), r, i, -1)
        cv2.imwrite(str(img_dir / f'syn_{k:03d}.png'), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
        cv2.imwrite(str(mask_dir / f'syn_{k:03d}.png'), mask)


paths = DS_CFG['paths']
if IN_KAGGLE and not Path(paths['images_dir']).exists():
    found_img, found_mask = find_dataset_dirs()
    if found_img is not None:
        paths.update(images_dir=str(found_img), masks_dir=str(found_mask))
        save_yaml(DS_CFG, DS_CFG_PATH)
        print('Tự phát hiện dữ liệu trên Kaggle:', found_img)

SYNTHETIC = not Path(paths['images_dir']).exists()
if SYNTHETIC:
    if not USE_SYNTHETIC_IF_MISSING:
        raise FileNotFoundError(f"Không thấy {paths['images_dir']} — sửa `paths` trong {DS_CFG_PATH}")
    syn_root = OUT_BASE / 'synthetic_data'
    if not (syn_root / 'images').exists():
        make_synthetic_dataset(syn_root)
    DS_CFG = {**DS_CFG, 'name': 'synthetic', 'format': 'instance_masks',
              'paths': {**paths, 'images_dir': str(syn_root / 'images'),
                        'masks_dir': str(syn_root / 'masks'), 'mask_suffix': ''}}
    paths = DS_CFG['paths']
    PROCESSED = syn_root / 'processed'
    display(Markdown('> ⚠️ **Không tìm thấy dữ liệu thật** → đang dùng **dữ liệu giả** để chạy thử notebook. '
                     'Sửa `paths` trong `configs/dataset.yaml` rồi chạy lại để có số liệu thật.'))

from src.data import load_records, load_records_from_config, load_splits, make_splits, save_records, save_splits


def load_records_and_splits(force=False):
    """Ưu tiên records.json/splits.json (đã chạy prepare_data.py); nếu chưa có thì đọc annotation gốc."""
    rp, sp = PROCESSED / 'records.json', PROCESSED / 'splits.json'
    if rp.exists() and sp.exists() and not force:
        return load_records(rp), load_splits(sp), 'records.json + splits.json'
    t0 = time.time()
    recs = load_records_from_config(DS_CFG)
    c = DS_CFG.get('split', {}) or {}
    spl = make_splits(recs, val=c.get('val', 0.15), test=c.get('test', 0.15), seed=c.get('seed', 42),
                      stratify_by_count=c.get('stratify_by_count', True))
    if SYNTHETIC:   # chỉ ghi cho dữ liệu giả; dữ liệu thật hãy chạy scripts/prepare_data.py
        save_records(recs, rp)
        save_splits(spl, sp)
    return recs, spl, f'annotation gốc ({time.time() - t0:.1f}s)'

In [ ]:
records, splits, source = load_records_and_splits(force=FORCE_RELOAD)
rec_by_id = {r.image_id: r for r in records}

# Chọn một ảnh đại diện có số quả từ trung bình đến nhiều
sorted_recs = sorted(records, key=lambda r: r.count, reverse=True)
sample_rec = sorted_recs[min(5, len(sorted_recs) - 1)]

print(f"Chọn ảnh mẫu: {sample_rec.image_id} với {sample_rec.count} quả")
print(f"Kích thước gốc: {sample_rec.width}x{sample_rec.height}")

from src.data.annotations import read_instance_mask
from src.data.augmentation import Sample

sample_img = load_image(sample_rec.image_path)
sample_mask = read_instance_mask(sample_rec.mask_path) if sample_rec.mask_path and Path(sample_rec.mask_path).exists() else None
base_sample = Sample(sample_img, sample_rec.boxes.copy(), sample_rec.instance_ids.copy(), sample_mask)

## 2. Trực quan hóa từng phép biến đổi riêng lẻ
Khảo sát tác động của từng Transform trong `src/data/augmentation.py`.

In [ ]:
from src.data.augmentation import (
    HFlip, VFlip, ColorJitter, GaussianBlur,
    RandomScaleCrop, ResizeLongest, RandomCropFixed
)
from src.visualization import draw_boxes, overlay_masks, draw_count_banner

transforms_to_test = [
    ("Gốc (Không biến đổi)", None),
    ("HFlip (Lật ngang p=1.0)", HFlip(p=1.0)),
    ("VFlip (Lật dọc p=1.0)", VFlip(p=1.0)),
    ("ColorJitter (Đổi màu/sáng)", ColorJitter(p=1.0, brightness=0.3, contrast=0.3, saturation=0.3, hue=15)),
    ("GaussianBlur (Làm mờ)", GaussianBlur(p=1.0, ksize=(7, 9))),
    ("RandomScaleCrop (Scale & Crop)", RandomScaleCrop(p=1.0, scale=(0.5, 0.8), ratio=(0.8, 1.2))),
    ("RandomCropFixed (512x512)", RandomCropFixed(size=512)),
    ("ResizeLongest (max_side=640)", ResizeLongest(max_side=640))
]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

rng = np.random.default_rng(42)

for idx, (title, tf) in enumerate(transforms_to_test):
    if tf is None:
        s = base_sample
    else:
        # copy sample để tránh mutate
        s_in = Sample(base_sample.image.copy(), base_sample.boxes.copy(), base_sample.ids.copy(),
                      None if base_sample.inst_mask is None else base_sample.inst_mask.copy())
        s = tf(s_in, rng)
    
    vis = s.image.copy()
    if s.inst_mask is not None and len(s.boxes):
        masks_bool = np.stack([s.inst_mask == i for i in s.ids]) if len(s.ids) else np.zeros((0, s.height, s.width), bool)
        vis = overlay_masks(vis, masks_bool, alpha=0.35)
    vis = draw_boxes(vis, s.boxes)
    vis = draw_count_banner(vis, f"Count: {len(s.boxes)} | {s.width}x{s.height}")
    
    axes[idx].imshow(vis)
    axes[idx].set_title(title, fontsize=11, fontweight='bold')
    axes[idx].axis('off')

plt.tight_layout()
savefig(fig, "single_transforms_demo")

## 3. Kiểm tra biến đổi đồng bộ (Ảnh - Bounding Box - Instance Mask)
Đặc biệt kiểm tra tính đúng đắn khi crop:
- Box và Mask có bị lệch tọa độ không?
- Khi quả bị cắt một phần ở viền crop, box có được thu nhỏ lại theo phần mask còn lại không?

In [ ]:
# Thực hiện RandomScaleCrop nhiều lần và phóng to vào các quả ở mép crop
crop_tf = RandomScaleCrop(p=1.0, scale=(0.4, 0.6), min_visibility=0.35)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i in range(6):
    s_in = Sample(base_sample.image.copy(), base_sample.boxes.copy(), base_sample.ids.copy(),
                  None if base_sample.inst_mask is None else base_sample.inst_mask.copy())
    cropped = crop_tf(s_in, np.random.default_rng(i + 10))
    
    vis = cropped.image.copy()
    if cropped.inst_mask is not None and len(cropped.ids):
        masks_bool = np.stack([cropped.inst_mask == inst_id for inst_id in cropped.ids])
        vis = overlay_masks(vis, masks_bool, alpha=0.4)
    vis = draw_boxes(vis, cropped.boxes)
    vis = draw_count_banner(vis, f"Crop #{i+1}: {len(cropped.boxes)} fruits")
    
    axes[i].imshow(vis)
    axes[i].set_title(f"Crop #{i+1} ({cropped.width}x{cropped.height} px)", fontsize=11)
    axes[i].axis('off')

plt.tight_layout()
savefig(fig, "synchronized_crop_verification")

## 4. Pipeline hoàn chỉnh: Faster R-CNN, Mask R-CNN và CSRNet
Sử dụng `build_augmenter` từ `configs/augmentation.yaml`.
- Faster / Mask R-CNN: `RandomScaleCrop` + `HFlip` + `ColorJitter` + `ResizeLongest(1024)`.
- CSRNet: `ResizeLongest` + `RandomCropFixed(512)` + `PadToMultiple(8)`.

In [ ]:
from src.data.augmentation import build_augmenter

aug_rcnn = build_augmenter(AUG_CFG, train=True, seed=123)
aug_csrnet = build_augmenter(AUG_CFG, train=True, seed=123, crop_size=CSRNET_CFG.get('train', {}).get('crop_size', 512), pad_multiple=8)
aug_val = build_augmenter(AUG_CFG, train=False)

s_rcnn = aug_rcnn(Sample(base_sample.image.copy(), base_sample.boxes.copy(), base_sample.ids.copy(),
                         None if base_sample.inst_mask is None else base_sample.inst_mask.copy()))

s_csr = aug_csrnet(Sample(base_sample.image.copy(), base_sample.boxes.copy(), base_sample.ids.copy(),
                          None if base_sample.inst_mask is None else base_sample.inst_mask.copy()))

s_val = aug_val(Sample(base_sample.image.copy(), base_sample.boxes.copy(), base_sample.ids.copy(),
                       None if base_sample.inst_mask is None else base_sample.inst_mask.copy()))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, s, title in zip(
    axes,
    [s_rcnn, s_csr, s_val],
    ["Faster / Mask R-CNN (Train Pipeline)", "CSRNet (Crop 512 + Pad8)", "Validation (Resize Only)"]
):
    vis = draw_boxes(s.image, s.boxes)
    vis = draw_count_banner(vis, f"Count: {len(s.boxes)} | Size: {s.width}x{s.height}")
    ax.imshow(vis)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
savefig(fig, "pipeline_comparison")

## 5. Sinh Density Map cho CSRNet sau Augmentation
Density map GT được tính toán từ tâm bounding box **sau khi** ảnh đã được biến đổi.
Quy tắc bảo toàn quan trọng:
$$\sum_{x, y} \text{DensityMap}(x, y) \approx N_{\text{fruits}}$$

In [ ]:
from src.data.density import boxes_to_density
from src.visualization import overlay_density

# Sinh density map cho ảnh mẫu sau khi qua pipeline của CSRNet
ds_ratio = DENSITY_CFG.get('downsample', 8)
dens_map = boxes_to_density(
    s_csr.boxes,
    s_csr.image.shape[:2],
    downsample=ds_ratio,
    sigma_scale=DENSITY_CFG.get('sigma_scale', 0.25),
    min_sigma=DENSITY_CFG.get('min_sigma', 1.0),
    fixed_sigma=DENSITY_CFG.get('fixed_sigma', None)
)

integrated_count = float(dens_map.sum())
gt_box_count = float(len(s_csr.boxes))

print(f"Số lượng box trong crop : {gt_box_count}")
print(f"Tổng tích phân density  : {integrated_count:.4f}")
print(f"Sai lệch tuyệt đối       : {abs(integrated_count - gt_box_count):.4f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

axes[0].imshow(draw_boxes(s_csr.image, s_csr.boxes))
axes[0].set_title(f"Ảnh + Bounding Box ({int(gt_box_count)} quả)")
axes[0].axis('off')

im_density = axes[1].imshow(dens_map, cmap='jet')
axes[1].set_title(f"Density Map GT (Downsample {ds_ratio}x, sum={integrated_count:.2f})")
axes[1].axis('off')
fig.colorbar(im_density, ax=axes[1], fraction=0.046, pad=0.04)

overlay_img = overlay_density(s_csr.image, dens_map, alpha=0.5)
axes[2].imshow(overlay_img)
axes[2].set_title("Chập Density Heatmap lên ảnh gốc")
axes[2].axis('off')

plt.tight_layout()
savefig(fig, "csrnet_density_gt_verification")

## 6. Khảo sát siêu tham số Sigma của Density Map
Ảnh hưởng của `sigma_scale` và `min_sigma` đối với sự tập trung / lan tỏa của Gaussian kernel.
Nếu `sigma` quá lớn, các quả đứng cạnh nhau sẽ hòa thành một khối lớn; nếu quá nhỏ, tín hiệu học bị thưa thớt (sparse).

In [ ]:
sigmas_to_test = [
    ("sigma_scale=0.15, min=0.5", 0.15, 0.5),
    ("sigma_scale=0.25, min=1.0 (Default)", 0.25, 1.0),
    ("sigma_scale=0.40, min=1.5", 0.40, 1.5),
    ("fixed_sigma=2.0", None, 2.0),
]

fig, axes = plt.subplots(1, len(sigmas_to_test), figsize=(20, 5))

for ax, (label, sc, ms) in zip(axes, sigmas_to_test):
    dm = boxes_to_density(
        s_csr.boxes,
        s_csr.image.shape[:2],
        downsample=8,
        sigma_scale=sc if sc else 0.25,
        min_sigma=ms,
        fixed_sigma=ms if sc is None else None
    )
    heat = overlay_density(s_csr.image, dm, alpha=0.5)
    ax.imshow(heat)
    ax.set_title(f"{label}\nSum={dm.sum():.2f}", fontsize=10)
    ax.axis('off')

plt.tight_layout()
savefig(fig, "density_sigma_comparison")

## 7. Thống kê biến đổi phân bố (Simulation N=100)
Mô phỏng áp dụng augmentation pipeline trên 100 lần để thống kê:
- Tỉ lệ số quả giữ lại sau khi Random Scale Crop.
- Phân bố kích thước quả sau augment.

In [ ]:
aug_stats = []
sim_rng = np.random.default_rng(2024)

# Thử nghiệm trên 20 ảnh khác nhau, mỗi ảnh augment 5 lần
test_sample_recs = sorted_recs[::max(1, len(sorted_recs)//20)][:20]

for r in test_sample_recs:
    im = load_image(r.image_path)
    orig_count = r.count
    for _ in range(5):
        s_raw = Sample(im, r.boxes.copy(), r.instance_ids.copy(), None)
        s_aug = aug_rcnn(s_raw)
        aug_count = len(s_aug.boxes)
        box_w = s_aug.boxes[:, 2] - s_aug.boxes[:, 0] if aug_count else np.array([])
        box_h = s_aug.boxes[:, 3] - s_aug.boxes[:, 1] if aug_count else np.array([])
        sides = np.sqrt(box_w * box_h) if aug_count else np.array([])
        
        aug_stats.append({
            'orig_count': orig_count,
            'aug_count': aug_count,
            'retention_rate': aug_count / max(1, orig_count),
            'mean_side': sides.mean() if len(sides) else np.nan
        })

df_sim = pd.DataFrame(aug_stats)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].hist(df_sim['retention_rate'] * 100, bins=25, color='#4C72B0', edgecolor='white')
axes[0].axvline((df_sim['retention_rate'] * 100).mean(), color='red', ls='--',
                label=f"TB: {(df_sim['retention_rate']*100).mean():.1f}%")
axes[0].set(title="Tỉ lệ quả giữ lại sau Random Scale Crop", xlabel="% số quả được giữ lại", ylabel="Số lần thử")
axes[0].legend()

axes[1].scatter(df_sim['orig_count'], df_sim['aug_count'], alpha=0.6, color='#55A868')
axes[1].plot([0, df_sim['orig_count'].max()], [0, df_sim['orig_count'].max()], 'r--', label='Đường y=x (Gốc)')
axes[1].set(title="Số quả gốc vs Số quả sau Augmentation", xlabel="Số quả gốc", ylabel="Số quả sau Augment")
axes[1].legend()

plt.tight_layout()
savefig(fig, "augmentation_simulation_stats")

display(df_sim.describe().round(2))

## 8. So sánh với Augmentation trong YOLO (Ultralytics)
Trích xuất cấu hình augmentation của YOLO trong `configs/models/yolov8.yaml`:
- Mosaic ($p=1.0$)
- Mixup, HSV Augmentation ($h=0.015, s=0.5, v=0.3$)
- Scale ($0.5$), Translate ($0.1$), Fliplr ($0.5$).

In [ ]:
yolo_cfg = load_yaml(ROOT / 'configs/models/yolov8.yaml')
train_args = yolo_cfg.get('train_args', {})

aug_params = {
    'Mosaic': train_args.get('mosaic', 0.0),
    'Close Mosaic (epochs)': train_args.get('close_mosaic', 0),
    'Horizontal Flip (fliplr)': train_args.get('fliplr', 0.0),
    'Vertical Flip (flipud)': train_args.get('flipud', 0.0),
    'Scale': train_args.get('scale', 0.0),
    'Translate': train_args.get('translate', 0.0),
    'HSV-Hue': train_args.get('hsv_h', 0.0),
    'HSV-Sat': train_args.get('hsv_s', 0.0),
    'HSV-Val': train_args.get('hsv_v', 0.0),
}

df_yolo_aug = pd.DataFrame(list(aug_params.items()), columns=['Phép biến đổi (Ultralytics)', 'Giá trị'])
display(df_yolo_aug)

print("Ghi chú: YOLO thực hiện mosaic bằng cách ghép 4 ảnh thành 1.")
print("Điều này giúp mô hình phát hiện tốt các quả ở tỉ lệ nhỏ và mật độ dày đặc.")

## 9. Tổng kết & Nhận xét
- Pipeline augmentation bằng numpy/OpenCV trong `src/data/augmentation.py` đã đảm bảo tính nhất quán chặt chẽ giữa ảnh, bounding box và instance mask.
- Phép sinh density map ground-truth cho CSRNet bảo toàn chính xác tổng tích phân số quả ($\sum D \approx N$).
- Việc crop cố định $512 \times 512$ ở CSRNet giúp mô hình huấn luyện ổn định theo batch với kích thước đồng nhất.
- Tiếp theo: Sang `03_train.ipynb` để thiết lập và điều phối huấn luyện.